# 工具封装

In [1]:
from langchain.tools import tool

c:\Users\houlj12\Desktop\work\test\LLM_demo\agent_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
@tool
def multiply(a:int, b:int) ->int:
    """2个数的乘积"""
    return a * b

print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

multiply
2个数的乘积
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
{'description': '2个数的乘积', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [ ]:
# 天气查询工具封装
import requests
def get_weather(city:str) ->str:
    """
    调用实时天气API，返回温度及天气状况参数:
    city:城市名称，如:'北京'
    """
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    payload = {"areacode" : "101010100"}
    headers = {
        "X-APISpace-Token":"xxx"
    }
    response=requests.request("GET", url, params=payload, headers=headers)
    return response.text

get_weather('北京')

'{"status":0,"result":{"location":{"areacode":"101010100","name":"北京","country":"中国","path":"北京,北京市,北京市,中国"},"realtime":{"text":"多云","code":"01","temp":16.5,"feels_like":15,"rh":48,"wind_class":"3级","wind_speed":3.8,"wind_dir":"西北风","wind_angle":332,"prec":0.0,"prec_time":"2026-04-19 15:00:00","clouds":60,"vis":13100,"pressure":1008,"dew":5,"uv":4,"weight":7,"brief":"凉","detail":"天气有点凉，出门不穿单"},"last_update":"2026-04-19 15:42"}}'

In [5]:
# 获取城市编码
import pandas as pd
from regex import match
city_df = pd.read_csv("city.csv")
def get_city_code(city_name:str) ->int:
    """
    获取城市编码
    """
    # 优先匹配区县
    match = city_df[city_df['district']==city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 匹配城市
    match = city_df[city_df['city']==city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 匹配省份
    match = city_df[city_df['province']==city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    return 101010100

get_city_code('丰台')


np.int64(101010900)

In [ ]:
# # 天气查询工具封装
# from urllib import response
# import requests
# from langchain.tools import tool

# @tool
# def get_weather(city:str) ->str:
#     """
#     调用实时天气API，返回温度及天气状况参数:
#     city:城市名称，如:'北京'
#     """
#     url = "https://eolink.o.apispace.com/456456/weather/v001/now"
#     city_code = get_city_code(city)
#     payload = {"areacode" : city_code}
#     headers = {
#         "X-APISpace-Token":"ij607k75z1dng5bjnr34byl6vfo3bxud"
#     }
#     response=requests.request("GET", url, params=payload, headers=headers)
#     #将结果转换为json数据
#     data = response.json()
#     temp = data.get('result').get('realtime').get('temp')
#     text = data.get('result').get('realtime').get('text')
#     return f"温度: {temp}°C, 天气: {text}"

# 调用工具

In [10]:
# 智普AI
key = 'cb5a18943da54ec389aa9fbd7e20e754.6vVc3r46yQXq7nCD'
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import os

os.environ["ZHIPUAI_API_KEY"] = key
chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5, #模型温度，值在0-1之间。值越小，随机性越低
)

In [13]:
import langchain
import langchain_core
print("langchain version:", langchain.__version__)
print("langchain_core version:", langchain_core.__version__)

langchain version: 1.2.15
langchain_core version: 1.3.0


In [ ]:
# 天气查询工具封装
from urllib import response
import requests
# from langchain.tools import tool - 新版本有修改！！！
from langchain_core.tools import tool

@tool
def get_weather(city:str) ->str:
    """
    调用实时天气API，返回温度及天气状况参数:
    city:城市名称，如:'北京'
    """
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    city_code = get_city_code(city)
    payload = {"areacode" : city_code}
    headers = {
        "X-APISpace-Token":"ij607k75z1dng5bjnr34byl6vfo3bxud"
    }
    response=requests.request("GET", url, params=payload, headers=headers)
    #将结果转换为json数据
    data = response.json()
    temp = data.get('result').get('realtime').get('temp')
    text = data.get('result').get('realtime').get('text')
    return f"温度: {temp}°C, 天气: {text}"

@tool
def multiply(a:int, b:int) ->int:
    """
    2个数的乘积，需要传递两个参数
    args:
        a: int
        b: int
    returns: int    
    """
    return a * b    

In [ ]:
from langgraph.prebuilt import create_react_agent


# # 创建工具对象
# tools = [get_weather]
# # 获取提示词
# prompt = hub.pull("hwchase17/react")
# # 创建智能体
# agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)
# # 创建AgentExecutor，运行智能体
# agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
# # 调用智能体
# response = agent_executor.invoke({"input": "今天成都天气如何?"})
# print(response)

# 1. 创建智能体（替代原 AgentExecutor）
#    - 不再需要手动获取提示词和创建 Agent，create_react_agent 内部已集成 ReAct 逻辑[citation:10]
#    - chat 是你已初始化的 LLM 实例（如 ChatOpenAI）
agent_executor = create_react_agent(
    model=chat,
    tools=[get_weather],
    prompt="You are a helpful assistant.",  # 可选，可替代 hub.pull
    debug=True  # 开启调试模式，会打印详细执行过程
)

# 2. 调用智能体（注意输入格式变化）
#    新版本使用 messages 列表格式[citation:3][citation:8]
response = agent_executor.invoke(
    {"messages": [("user", "今天成都天气如何?")]}
)

# 3. 打印最终回答（注意输出结构变化）
print(response["messages"][-1].content)

C:\Users\houlj12\AppData\Local\Temp\ipykernel_15052\3966665867.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


[values] {'messages': [HumanMessage(content='今天成都天气如何?', additional_kwargs={}, response_metadata={}, id='1b6d6767-1780-4996-9693-bd549b9267a3')]}
[updates] {'agent': {'messages': [AIMessage(content='\n我来帮您查询成都今天的天气情况。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"city":"成都"}', 'name': 'get_weather'}, 'id': 'call_-7696267586941481408', 'index': 0, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 188, 'total_tokens': 212}, 'model_name': 'glm-4', 'finish_reason': 'tool_calls'}, id='lc_run--019da5cd-cdb3-77e3-9a58-c2cfc43b243e-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '成都'}, 'id': 'call_-7696267586941481408', 'type': 'tool_call'}], invalid_tool_calls=[])]}}
[values] {'messages': [HumanMessage(content='今天成都天气如何?', additional_kwargs={}, response_metadata={}, id='1b6d6767-1780-4996-9693-bd549b9267a3'), AIMessage(content='\n我来帮您查询成都今天的天气情况。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{

In [31]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(
    model=chat,
    tools=[get_weather, multiply],
    prompt="You are a helpful assistant.",  # 可选，可替代 hub.pull
    debug=True  # 开启调试模式，会打印详细执行过程
)

response = agent_executor.invoke(
    {"messages": [("user", "1x3等于几?")]}
)

# 3. 打印最终回答（注意输出结构变化）
print(response["messages"][-1].content)

C:\Users\houlj12\AppData\Local\Temp\ipykernel_15052\64282763.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


[values] {'messages': [HumanMessage(content='1x3等于几?', additional_kwargs={}, response_metadata={}, id='dc2d40e3-aaa9-48b9-a21b-f080f032f862')]}
[updates] {'agent': {'messages': [AIMessage(content='\n我来帮您计算1乘以3的结果。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":1,"b":3}', 'name': 'multiply'}, 'id': 'call_-7696240236589739852', 'index': 0, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 272, 'total_tokens': 304}, 'model_name': 'glm-4', 'finish_reason': 'tool_calls'}, id='lc_run--019da5d8-a933-7f30-a0d4-68b7eae55ece-0', tool_calls=[{'name': 'multiply', 'args': {'a': 1, 'b': 3}, 'id': 'call_-7696240236589739852', 'type': 'tool_call'}], invalid_tool_calls=[])]}}
[values] {'messages': [HumanMessage(content='1x3等于几?', additional_kwargs={}, response_metadata={}, id='dc2d40e3-aaa9-48b9-a21b-f080f032f862'), AIMessage(content='\n我来帮您计算1乘以3的结果。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":1,"b":3}

In [32]:
test_inputs = ["今天北京天气如何?","2x3等于几?","帮我修修电脑"]
for query in test_inputs:
    response = agent_executor.invoke(
        {"messages": [("user", query)]}
    )
    print(f"输入: {query}\n回答: {response['messages'][-1].content}\n{'-'*50}")

[values] {'messages': [HumanMessage(content='今天北京天气如何?', additional_kwargs={}, response_metadata={}, id='9ba007d2-c54f-49cf-8e14-65058c73f3f1')]}
[updates] {'agent': {'messages': [AIMessage(content='\n我来帮您查询今天北京的天气情况。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"city":"北京"}', 'name': 'get_weather'}, 'id': 'call_-7696195019174050030', 'index': 0, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 271, 'total_tokens': 295}, 'model_name': 'glm-4', 'finish_reason': 'tool_calls'}, id='lc_run--019da5de-4b3f-7f73-9758-ec0c749beefd-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7696195019174050030', 'type': 'tool_call'}], invalid_tool_calls=[])]}}
[values] {'messages': [HumanMessage(content='今天北京天气如何?', additional_kwargs={}, response_metadata={}, id='9ba007d2-c54f-49cf-8e14-65058c73f3f1'), AIMessage(content='\n我来帮您查询今天北京的天气情况。\n', additional_kwargs={'tool_calls': [{'function': {'arguments': '{